# Task 1 — определение компаний, отрасли и цитируемости

In [15]:
from pathlib import Path
from urllib.parse import urlparse
from collections import Counter
import html
import re
import time
import json
import warnings

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics import accuracy_score, f1_score, adjusted_rand_score, classification_report

MAX_ARTICLES_PER_EVENT = 4
RANDOM_STATE = 42
INPUT_FILE = Path("data/news.csv")
OUTPUT_FILE = Path("news_predictions.csv")
warnings.filterwarnings("ignore")

## 1. Отрасли и общие правила отбора компаний

In [16]:
INDUSTRIES = {
    "экономика": "финансовые рынки, банки, налоги, инвестиции, доходы и экономические показатели",
    "политика": "государственная политика, выборы, международные отношения, законы и безопасность",
    "недвижимость": "рынок недвижимости, покупка и аренда квартир, ипотека и стоимость жилья",
    "транспорт": "перевозки пассажиров и грузов, авиация, железные дороги, автомобили и аэропорты",
    "промышленность": "промышленное производство, машиностроение, фабрики, заводы и оборудование",
    "потребительский сектор": "торговля, магазины, товары, услуги, рестораны и потребительский спрос",
    "металлургия": "производство стали, алюминия, меди, металлических сплавов и добыча металлической руды",
    "нефтегаз": "добыча нефти и природного газа, нефтепереработка, газопроводы и нефтепродукты",
    "телекоммуникации": "мобильная связь, операторы связи, телекоммуникационные сети и интернет-провайдеры",
    "энергетика": "электростанции, электроэнергия, энергосети, генерация и снабжение электричеством",
    "строительство": "строительные работы, возведение объектов, подрядчики и строительные проекты",
    "сельское хозяйство": "выращивание сельскохозяйственных культур, урожай, фермеры и животноводство",
    "наука": "научные исследования, открытия, лаборатории, эксперименты и работа ученых",
    "экология": "загрязнение окружающей среды, охрана природы, климат, отходы и выбросы",
}

NON_COMPANY_TYPE = re.compile(
    r"\b(?:министерств\w*|ведомств\w*|госдум\w*|правительств\w*|"
    r"прокуратур\w*|полици\w*|администраци\w*|департамент\w*|"
    r"федеральн\w*\s+служб\w*|управлени\w*\s+мвд|"
    r"следственн\w*\s+комитет\w*|совет\s+федерац\w*|"
    r"международн\w*\s+организац\w*|вооруженн\w*\s+сил\w*|"
    r"центральн\w*\s+банк\w*|банк\s+росси\w*|"
    r"университет\w*|академи\w*|институт\w*|больниц\w*|"
    r"федерац\w*\s+спорт\w*|общественн\w*\s+объединен\w*|"
    r"информационн\w*\s+агентств\w*|газет\w*|журнал\w*|"
    r"редакци\w*|издан\w*|телеканал\w*|радиостанц\w*|"
    r"пресс-служб\w*|сил\w*\s+(?:внутренн\w*\s+)?безопасност\w*|"
    r"сми|интернет-портал\w*)\b", re.I
)

PUBLIC_ACRONYMS = {
    "мвд", "мчс", "фсб", "фнс", "фас", "ск", "пво", "всу", "вкс",
    "минфин", "минобороны", "минюст", "росстат", "роскомнадзор",
    "госдума", "цб", "оон", "нато", "воз", "юнеско", "мок", "ран",
}

MEDIA_NAMES = {
    "тасс", "интерфакс", "риа новости", "белта", "reuters", "рейтер",
    "bloomberg", "блумберг", "life.ru", "rbc", "рбк", "сми",
}

LEGAL_PREFIX = re.compile(
    r"^(?:(?:ПАО|АО|ООО|ОАО|ЗАО|ГК|группа компаний|компания|корпорация|холдинг)\s+)+",
    re.I,
)

LEGAL_NAME = re.compile(
    r"(?i:\b(?:ПАО|АО|ООО|ОАО|ЗАО|компания|корпорация|холдинг|"
    r"авиакомпания|производитель))\s+[«\"“]?"
    r"([A-ZА-ЯЁ][\w&.-]*(?:\s+[A-ZА-ЯЁ][\w&.-]*){0,3})",
)

LATIN_NAME = re.compile(r"(?<!\w)([A-Z][A-Za-z0-9&.-]{2,}(?:\s+[A-Z][A-Za-z0-9&.-]{2,}){0,2})")

SOURCE_INTRO = re.compile(
    r"(?:сообща\w*|переда\w*|пиш\w*|информиру\w*|по данным|по информации|"
    r"со ссылкой на|источник\s*:|как уточня\w*|корреспондент\w*|"
    r"материал\w*\s+опубликован\w*\s+на)\s*(?:агентств\w*|издан\w*|портал\w*|"
    r"газет\w*|сайт\w*|телеканал\w*|СМИ)?\s*$", re.I
)

BUSINESS_CUE = re.compile(
    r"(?:компани\w*|корпораци\w*|холдинг\w*|производител\w*|"
    r"авиакомпани\w*|ретейлер\w*|оператор\w*|девелопер\w*|"
    r"предприяти\w*|ПАО|ООО|АО|ОАО|ЗАО)\s*[«\"“]?\s*$", re.I
)

LOCATION_AFTER_PREPOSITION = re.compile(
    r"(?i:\b(?:в|на|из|под|около|возле)\s+)"
    r"(?:(?i:городе|городом|посёлке|поселке|аэропорту|селе|районе|"
    r"республике|области|промышленной\s+зоне)\s+)?"
    r"([А-ЯЁ][а-яё-]{2,}(?:-[А-ЯЁ][а-яё-]{2,})?)\b"
)

INCIDENT = re.compile(
    r"\b(?:взрыв\w*|пожар\w*|возгорани\w*|авари\w*|крушени\w*|"
    r"столкновени\w*|катастроф\w*|дтп|посадк\w*|"
    r"ураган\w*|торнадо|смерч\w*|утечк\w*|обрушени\w*)\b", re.I
)

CAUSE = re.compile(
    r"\b(?:причин\w*|из-за|вызван\w*|последстви\w*|установил\w*|"
    r"выяснил\w*|расследовани\w*|почему|что привело)\b", re.I
)

DATELINE_TITLE = re.compile(
    r"^\s*(?:\d{1,2}\s+[а-яё]+\s*,\s*[А-ЯЁ][а-яё-]+\s*/\s*корр\.|"
    r"[А-ЯЁ][а-яё-]+\.\s*|[А-ЯЁ]{2,},\s*\d{1,2}\s+[а-яё]+\.?\s*/)",
    re.I,
)

DATELINE_IN_BODY = re.compile(
    r"^\s*(?:[А-ЯЁ][А-ЯЁа-яё-]{2,30}[,.]\s*)?\d{1,2}\s+[а-яё]+[,.]\s*"
    r"(?:/\s*)?[А-ЯЁA-Z]{3,25}\s*(?:/\s*\.?|[-–])\s*"
)

# Универсальные признаки госоргана и СМИ
EXCLUDED_ORG_WORDS = re.compile(
    r"\b(?:госавтоинспекц\w*|социальн\w*\s+фонд\w*|"
    r"минздрав\w*|роспотребнадзор\w*|умвд|отдел\w*\s+полици\w*|"
    r"телеграм[-\s]*канал\w*|telegram[-\s]*канал\w*|"
    r"социальн\w*\s+сет\w*|федеральн\w*\s+антимонопольн\w*\s+служб\w*|"
    r"известия|комсомольск\w*\s+правд\w*|новостн\w*\s+агентств\w*|"
    r"информационн\w*\s+портал\w*)\b", re.I)

In [17]:
def clean_text(value):
    value = html.unescape(str(value if pd.notna(value) else ""))
    value = re.sub(r"https?://\S+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

def prepare_text(title, body):
    """Remove duplicates and datelines; never cluster by a metadata headline."""
    title, body = clean_text(title), clean_text(body)
    if title and body.casefold().startswith(title.casefold()):
        body = body[len(title):].strip()

    body = DATELINE_IN_BODY.sub("", body, count=1).strip()

    if len(title) < 18 or DATELINE_TITLE.match(title):
        body = body.strip(" /.-")
        body = re.sub(r"^\s*[А-ЯЁA-Z]{3,25}\s*[/.-]+\s*", "", body)
        if body:
            sentence = re.split(r"(?<=[.!?])\s+(?=[А-ЯЁA-Z])", body, maxsplit=1)[0]
            title = sentence if 25 <= len(sentence) <= 220 else body[:180].rsplit(" ", 1)[0]
    return title, body

def publisher_from_source(source):
    source = str(source or "").strip()
    if not source:
        return ""
    parsed = urlparse(source if "://" in source else "https://" + source)
    host = (parsed.hostname or "").lower().removeprefix("www.")
    if host in {"t.me", "telegram.me"}:
        parts = [part for part in parsed.path.split("/") if part and part != "s"]
        return "telegram:" + (parts[0].lower() if parts else source.lower())
    if not host:
        return "unknown:" + source.lower()
    try:
        import tldextract
        extracted = tldextract.TLDExtract(suffix_list_urls=())(host)
        return (f"{extracted.domain}.{extracted.suffix}" if extracted.suffix else host)
    except ImportError:
        return ".".join(host.split(".")[-2:])

def load_ner():
    from natasha import Doc, MorphVocab, NewsEmbedding, NewsNERTagger, Segmenter
    return Doc, Segmenter(), NewsNERTagger(NewsEmbedding()), MorphVocab()

def load_embedding_model():
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def load_industry_classifier(accurate=False, device=None):
    """NLI натренирован класссифицировать события"""
    from transformers import pipeline
    model_name = (
        "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli" if accurate
        else "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli"
    )
    options = {"device": device} if device is not None else {}
    classifier = pipeline("zero-shot-classification", model=model_name, **options)

    if classifier.device.type == "cpu":
        classifier.model.float()

    return classifier

def extract_entities(title, body, ner):
    """Возвращает организации и их позиции, а также локации"""
    if ner is None:
        return [], set()
    Doc, segmenter, tagger, vocab = ner
    doc = Doc(f"{title}. {body[:1000]}")
    doc.segment(segmenter)
    doc.tag_ner(tagger)
    organizations, locations = [], set()
    for span in doc.spans:
        span.normalize(vocab)
        name = clean_text(span.normal or span.text).strip(' «»".,;:')
        if span.type == "ORG":
            organizations.append((name, span.start, span.stop))
        elif span.type == "LOC" and span.start < len(title) + 2:
            locations.add(name.casefold())
    return organizations, locations

def compact_name(name):
    name = clean_text(name).strip(' «»"“”.,;:')
    name = LEGAL_PREFIX.sub("", name).strip(' «»"“”.,;:')
    return name

def is_non_company(name):
    key = name.casefold().replace("ё", "е").strip()
    return (not name or len(name) < 3 or key in PUBLIC_ACRONYMS or
            key in MEDIA_NAMES or bool(NON_COMPANY_TYPE.search(name)) or
            not re.search(r"[A-Za-zА-Яа-яЁё]", name))

def is_attributed_source(text, start):
    """True для выражений 'по данным ...', 'пишет ...', 'источник: ...'."""
    prefix = text[max(0, start - 90):start]
    return bool(SOURCE_INTRO.search(prefix))

def is_same_as_publisher(name, publisher):
    """Выделяет имя сайта даже если NER маркирует их как организации."""
    if not publisher or publisher.startswith("unknown:"):
        return False
    name_key = re.sub(r"\W", "", name.casefold())
    host_key = publisher.split(":")[-1].split(".")[0]
    host_key = re.sub(r"\W", "", host_key.casefold())
    return len(host_key) >= 3 and (name_key == host_key or name_key == host_key + "ru")

def find_companies(title, body, organizations, publisher=""):
    """Ищет компании, участвующие в истории, а не каждую организацию.

    NER предлагает кандидатов. Общие типы организаций, атрибуция публикаций,
    и URL-адрес издателя помогают отсеять ложные срабатывания. Неопределенные случаи остаются
    пустыми, а не превращаются из агентства в компанию.
    """
    sample = f"{title}. {body[:1000]}"
    candidates = [(name, start, end, "ner") for name, start, end in organizations]

    for match in LEGAL_NAME.finditer(sample):
        candidates.append((match.group(1), match.start(1), match.end(1), "legal"))
    for match in LATIN_NAME.finditer(title):
        candidates.append((match.group(1), match.start(1), match.end(1), "latin"))

    found = {}
    for raw_name, start, _, origin in candidates:
        name = compact_name(raw_name)
        if is_non_company(name) or is_same_as_publisher(name, publisher):
            continue
        before = sample[max(0, start - 75):start]
        in_title = start < len(title)
        source_context = is_attributed_source(sample, start)
        company_context = bool(BUSINESS_CUE.search(before[-65:]))
        if source_context and not company_context:
            continue

        if origin == "latin" and not in_title:
            continue
        score = (4 if in_title else 1) + (2 if origin == "ner" else 0)
        score += 4 if origin == "legal" or company_context else 0
        score -= 3 if source_context else 0
        if origin == "latin" and not company_context:
            score -= 1
        if score < 3:
            continue

        key = re.sub(r"\s+", " ", name.casefold().replace("ё", "е"))
        if key not in found or found[key][0] < score:
            found[key] = (score, start, name)

    return [x[2] for x in sorted(found.values(), key=lambda x: (-x[0], x[1]))[:3]]

def event_state(title):
    """Оформляет введение ограничения и его отмену как отдельные события."""
    start = bool(re.search(r"\b(?:ввел\w*|введен\w*|начал\w*|закрыл\w*|ограничил\w*)", title, re.I))
    end = bool(re.search(r"\b(?:отменил\w*|снял\w*|возобновил\w*|восстановил\w*)", title, re.I))
    if start and not end:
        return "start"
    if end and not start:
        return "end"
    return None

def title_locations(title, ner_locations):
    if ner_locations:
        return {x.casefold() for x in ner_locations}
    return {x.group(1).casefold() for x in LOCATION_AFTER_PREPOSITION.finditer(title)}

def locations_conflict(a, b):
    if not a or not b:
        return False
    return not any(x == y or (len(x) > 5 and len(y) > 5 and x[:5] == y[:5])
                   for x in a for y in b)

def incident_kind(text):
    match = INCIDENT.search(text)
    if not match:
        return ""
    word = match.group().casefold()
    if re.match(r"(?:взрыв)", word):
        return "explosion"
    if re.match(r"(?:пожар|возгоран)", word):
        return "fire"
    if re.match(r"(?:ураган|торнадо|смерч)", word):
        return "storm"
    if re.match(r"(?:авари|крушени|катастроф|столкновени|дтп)", word):
        return "accident"
    if re.match(r"(?:посадк)", word):
        return "landing"
    return word[:5]

def event_groups(titles, bodies, locations, model, neighbors=35):
    """Группирует отчеты об одном и том же событии, включая объяснения его причин.
    Использует семантическое сходство для предложения вариантов, затем проверяет совпадение фактов.
    """
    n = len(titles)
    if n < 2:
        return np.ones(n, dtype=int)

    head_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
    body_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)
    heads = head_tfidf.fit_transform(titles)
    leads = body_tfidf.fit_transform([b[:650] or t for t, b in zip(titles, bodies)])
    semantic = None
    if model is not None:
        semantic = np.asarray(model.encode(
            [f"{t}. {b[:350]}" for t, b in zip(titles, bodies)],
            normalize_embeddings=True, batch_size=32, show_progress_bar=False))

    phases = [event_state(t) for t in titles]
    incidents = [incident_kind(f"{t}. {b[:230]}") for t, b in zip(titles, bodies)]
    cause_reports = [bool(CAUSE.search(t)) for t in titles]
    pairs = set()
    for features in ([heads, leads] + ([semantic] if semantic is not None else [])):
        neighbors_model = NearestNeighbors(
            n_neighbors=min(neighbors + 1, n), metric="cosine", algorithm="brute")
        neighbors_model.fit(features)
        _, results = neighbors_model.kneighbors(features)
        for i, indices in enumerate(results):
            for j in indices:
                if i != j:
                    pairs.add((min(i, int(j)), max(i, int(j))))

    if not pairs:
        return np.arange(1, n + 1)
    pairs = np.array(sorted(pairs), dtype=int)
    left, right = pairs[:, 0], pairs[:, 1]
    hs = np.asarray(heads[left].multiply(heads[right]).sum(axis=1)).ravel()
    bs = np.asarray(leads[left].multiply(leads[right]).sum(axis=1)).ravel()
    ss = np.sum(semantic[left] * semantic[right], axis=1) if semantic is not None else np.zeros(len(pairs))

    links = []
    for k, (i, j) in enumerate(pairs):
        head, body, meaning = float(hs[k]), float(bs[k]), float(ss[k])
        if phases[i] and phases[j] and phases[i] != phases[j]:
            continue
        same_incident = bool(incidents[i] and incidents[i] == incidents[j])
        cause_pair = same_incident and (cause_reports[i] or cause_reports[j])
        conflict = locations_conflict(locations[i], locations[j])
        strong_duplicate = head >= 0.90 or body >= 0.92

        if model is not None:
            ordinary = (head >= 0.84 or body >= 0.88 or
                        (meaning >= 0.82 and head >= 0.38 and body >= 0.25) or
                        (meaning >= 0.88 and (head >= 0.28 or body >= 0.42)))
            follow_up = (cause_pair and meaning >= 0.72 and head >= 0.20 and body >= 0.20)
        else:
            ordinary = head >= 0.84 or body >= 0.88 or (head >= 0.62 and body >= 0.43)
            follow_up = cause_pair and head >= 0.46 and body >= 0.37

        if conflict and not strong_duplicate:
            if not (follow_up and (head >= 0.40 or body >= 0.48)):
                continue
        if ordinary or follow_up:
            score = 0.45 * head + 0.25 * body + 0.30 * meaning
            links.append((score, int(i), int(j), bool(follow_up)))

    parent = list(range(n))
    members = {i: {i} for i in range(n)}

    def root(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def compatible(a, b, follow_up):
        for x in members[a]:
            for y in members[b]:
                if phases[x] and phases[y] and phases[x] != phases[y]:
                    return False
                same_incident = bool(incidents[x] and incidents[x] == incidents[y])
                if locations_conflict(locations[x], locations[y]):
                    if not (follow_up and same_incident and
                            float(heads[x].multiply(heads[y]).sum()) >= 0.40):
                        return False
                head = float(heads[x].multiply(heads[y]).sum())
                lead = float(leads[x].multiply(leads[y]).sum())
                if head < (0.10 if follow_up and same_incident else 0.16) and lead < 0.16:
                    return False
        return True

    for _, i, j, follow_up in sorted(links, reverse=True):
        a, b = root(i), root(j)
        if a == b or not compatible(a, b, follow_up):
            continue
        if len(members[a]) < len(members[b]):
            a, b = b, a
        parent[b] = a
        members[a].update(members.pop(b))

    label_by_root = {}
    labels = []
    for i in range(n):
        r = root(i)
        if r not in label_by_root:
            label_by_root[r] = len(label_by_root) + 1
        labels.append(label_by_root[r])
    return np.asarray(labels)

## 2. Подключение моделей

In [18]:
def build_models(original):
    import natasha
    import sentence_transformers
    import transformers
    embedder = load_embedding_model()
    classifier = load_industry_classifier(accurate=False, device="cpu")
    ner = load_ner()
    return embedder, classifier, ner

## 3. Характер события, несколько сообщений и одна отрасль на группу

In [19]:
EVENT_EXAMPLES = {
    "экономика": [
        "Акционеры одобрили дивиденды, выросла капитализация компании и цена акций.",
        "Банк выдал кредит, изменились процентные ставки и финансовые показатели компании.",
    ],
    "политика": [
        "Правительство внесло законопроект, парламент принял закон и ввёл новые правила.",
        "Политик объявил об отставке, прошли выборы и дипломатические переговоры.",
    ],
    "недвижимость": [
        "Подорожали квартиры и аренда жилья, изменились условия ипотеки.",
        "Застройщик начал продажи жилого комплекса на рынке недвижимости.",
    ],
    "транспорт": [
        "Пассажирский самолёт совершил вынужденную посадку, рейс задержали.",
        "В аэропорту ограничили полёты, поезд отменили, изменились грузовые перевозки.",
    ],
    "промышленность": [
        "На заводе запустили линию производства и новое оборудование.",
        "Предприятие увеличило выпуск продукции, произошла авария на производстве.",
    ],
    "потребительский сектор": [
        "Торговая сеть открыла магазины и изменила цены на потребительские товары.",
        "Ретейлер сообщил о продажах, спросе покупателей и работе ресторанов.",
    ],
    "металлургия": [
        "Металлургический комбинат увеличил выплавку стали и алюминия.",
        "Предприятие запустило производство металлов и переработку руды.",
    ],
    "нефтегаз": [
        "На объекте нефтегазовой отрасли произошла авария, выясняют её причину.",
        "Компания нарастила добычу нефти и газа, начала переработку углеводородов.",
    ],
    "телекоммуникации": [
        "Оператор связи запустил мобильную сеть и новый тариф для абонентов.",
        "Интернет-провайдер расширил покрытие сети и восстановил связь.",
    ],
    "энергетика": [
        "Электростанция увеличила выработку электроэнергии и мощность генерации.",
        "На электросети произошла авария, восстановили подачу электричества.",
    ],
    "строительство": [
        "Подрядчик начал строить здание, мост и инфраструктурный объект.",
        "Строительная компания завершила работы и ввела объект в эксплуатацию.",
    ],
    "сельское хозяйство": [
        "Фермеры собрали урожай зерна и начали посевную кампанию.",
        "Сельхозпредприятие увеличило производство молока и мяса.",
    ],
    "наука": [
        "Учёные опубликовали исследование и сделали научное открытие.",
        "Исследователи провели эксперимент и представили результаты испытаний.",
    ],
    "экология": [
        "Обнаружено загрязнение реки и выбросы вредных веществ в атмосферу.",
        "Экологи оценили последствия разлива отходов и ущерб природе.",
    ],
}

In [20]:
# Общие признаки действий
ACTION_RULES = {
    "экономика": re.compile(r"\b(?:дивиденд\w*|капитализац\w*|бирж\w*|акци[йи]\b|котиров\w*|облигаци\w*|купон\w*|кредитн\w*\s+рейтинг\w*|дефолт\w*|ключев\w*\s+ставк\w*|инфляци\w*|вклад\w*|финансов\w*\s+отчет\w*)\b", re.I),
    "политика": re.compile(r"\b(?:законопроект\w*|депутат\w*|парламент\w*|санкци\w*|министр\w*|отставк\w*|выбор\w*|правительств\w*|указ\w*|дипломат\w*)\b", re.I),
    "нефтегаз": re.compile(r"\b(?:нефт\w*|газопровод\w*|газоперераб\w*|нефтеперераб\w*|спг|углеводород\w*|буров\w*|газов\w*\s+месторождени\w*)\b", re.I),
    "транспорт": re.compile(r"\b(?:самол[её]т\w*|авиарейс\w*|пассажирск\w*\s+рейс\w*|аэропорт\w*|железнодорож\w*|авиакомпани\w*|поезд\w*)\b", re.I),
}

def short_article(title, body):
    # Заголовок важен: обычно содержит именно произошедшее действие.
    return f"{title}. {body[:350]}".strip()


def classify_event_groups(news, titles, bodies, classifier, embeddings,
                          batch_size=6, max_articles=MAX_ARTICLES_PER_EVENT):
    labels = list(INDUSTRIES)
    groups = news.groupby("event_id", sort=False).indices
    group_ids = list(groups)
    if not group_ids:
        return {}

    article_texts = [short_article(t, b) for t,b in zip(titles,bodies)]
    vectors = np.asarray(embeddings.encode(article_texts, normalize_embeddings=True, batch_size=32))
    publisher_ids = news["_publisher"].tolist()

    selected = {}
    for group_id, indices in groups.items():
        candidates = sorted(map(int, indices), key=lambda i: len(titles[i]), reverse=True)
        chosen = [candidates.pop(0)]
        while candidates and len(chosen) < max_articles:
            # Добавляем разнообразную формулировку, по возможности из другого источника.
            def novelty(i):
                maximum = max(float(vectors[i] @ vectors[j]) for j in chosen)
                other_publisher = 0.05 if publisher_ids[i] not in {publisher_ids[j] for j in chosen} else 0
                return 1 - maximum + other_publisher
            next_i = max(candidates, key=novelty)
            if max(float(vectors[next_i] @ vectors[j]) for j in chosen) > 0.985:
                break
            chosen.append(next_i)
            candidates.remove(next_i)
        selected[group_id] = chosen

    chosen_indices = [i for gid in group_ids for i in selected[gid]]
    samples = [article_texts[i] for i in chosen_indices]
    selected_vectors = vectors[chosen_indices]
    ref_vectors = np.asarray(embeddings.encode(
        [f"Новость об отрасли {label}: {INDUSTRIES[label]}" for label in labels],
        normalize_embeddings=True))
    semantic = selected_vectors @ ref_vectors.T

    examples = [example for label in labels for example in EVENT_EXAMPLES[label]]
    event_vectors = np.asarray(embeddings.encode(examples, normalize_embeddings=True))
    title_vectors = np.asarray(embeddings.encode([titles[i] for i in chosen_indices],normalize_embeddings=True))
    event_similarity = (title_vectors @ event_vectors.T).reshape(len(samples),len(labels),2).max(axis=2)

    nli = None
    if classifier is not None:
        scores=[]
        for start in range(0,len(samples),batch_size):
            out = classifier(samples[start:start+batch_size], candidate_labels=labels,
                             hypothesis_template="Эта новость относится к теме «{}».",
                             multi_label=True, batch_size=2)
            if isinstance(out,dict): out=[out]
            for row in out:
                by_label=dict(zip(row['labels'],row['scores']))
                scores.append([by_label[label] for label in labels])
        nli=np.asarray(scores,dtype=float)

    def scale_rows(x):
        x=np.asarray(x,dtype=float)
        return (x-x.min(axis=1,keepdims=True))/np.maximum(np.ptp(x,axis=1,keepdims=True),1e-9)

    if nli is None:
        combined=0.67*scale_rows(semantic)+0.33*scale_rows(event_similarity)
    else:
        combined=0.50*scale_rows(semantic)+0.30*scale_rows(nli)+0.20*scale_rows(event_similarity)

    # Бонусы задаются типу события, а не названию организации.
    for k,i in enumerate(chosen_indices):
        head=titles[i]
        for category, pattern in ACTION_RULES.items():
            if pattern.search(head):
                bonus = 0.17 if category in {'экономика','политика'} else 0.10
                combined[k,labels.index(category)] += bonus

    predictions={}
    offset=0
    for group_id in group_ids:
        count=len(selected[group_id]); scores=combined[offset:offset+count].mean(axis=0); offset+=count
        best=int(np.argmax(scores))
        predictions[group_id]=labels[best]
    return predictions

## 4. Запуск по `news.csv`

In [24]:
def find_news_companies(title, body, organizations, publisher):
    candidates = find_companies(title,body,organizations,publisher)
    cleaned=[]
    for name in candidates:
        if EXCLUDED_ORG_WORDS.search(name) or is_non_company(name):
            continue
        if name not in cleaned:
            cleaned.append(name)
    return cleaned[:3]


def predict_from_original(original, embeddings, classifier, ner):
    required=['title','text','source']
    news=original[required].copy().fillna('').reset_index(drop=True)
    prepared=[prepare_text(t,b) for t,b in zip(news.title,news.text)]
    titles=[t for t,_ in prepared]; bodies=[b for _,b in prepared]
    publishers=[publisher_from_source(s) or f'unknown:{i}' for i,s in enumerate(news.source)]
    news['_publisher']=publishers
    found=[extract_entities(t,b,ner) for t,b in prepared]
    companies=[find_news_companies(t,b,org,publishers[i])
               for i,((t,b),(org,_)) in enumerate(zip(prepared,found))]
    places=[title_locations(t,loc) for t,(_,loc) in zip(titles,found)]
    news['event_id']=event_groups(titles,bodies,places,embeddings)
    # Группы предсказываются без доступа к эталону.
    industry_map=classify_event_groups(news,titles,bodies,classifier,embeddings)
    news['company']=[c[0] if c else 'Не указана' for c in companies]
    news['companies']=['; '.join(c) for c in companies]
    news['industry']=news.event_id.map(industry_map)
    news['citation_count']=news.groupby('event_id')['_publisher'].transform('nunique')
    news['articles_count']=news.groupby('event_id')['_publisher'].transform('size')
    result=news[required+['company','companies','industry','event_id',
                          'citation_count','articles_count']].copy()
    assert list(result.columns)==required+['company','companies','industry','event_id','citation_count','articles_count']
    assert len(result)==len(original)
    assert result.industry.notna().all()
    assert result.groupby('event_id').industry.nunique().max()==1
    return result

original=pd.read_csv(INPUT_FILE).fillna('')
start=time.time()
embedder,classifier,ner=build_models(original)
result=predict_from_original(original,embedder,classifier,ner)
result.to_csv(OUTPUT_FILE,index=False)
print(result[['title','company','industry']].head(6).to_string(index=False,max_colwidth=65))

Device set to use cpu


                                                            title            company         industry
Начался пожар после взрыва на крупнейшем в мире комплексе по п...         Не указана         нефтегаз
                   Отдых россиян в Азии снизил инфляцию в России.         Не указана        экономика
Правительство подготовило новую версию законопроекта об искусс...               Сбер            наука
                       ✨ Закрыт сбор на мобильные огневые группы!         Не указана телекоммуникации
↖️ Индия, один из ключевых покупателей российского топлива, ув...         Не указана         нефтегаз
Wildberries &amp; Russ отложила планы по развитию направления ... Wildberries & Russ            наука
